# Module 2.2: Adopting `USW-Enterprise-8-PoE`, Port Telemetry & PoE Safety Hygiene

## UniFi Storage & Enterprise Networking (Chapter 3) - Step 2: Managed Switch Control Plane

## 🎯 Learning Goals

By completing this laboratory module, you will be able to:

1. Adopt the **`USW-Enterprise-8-PoE`** (`10.0.0.203`) into your self-hosted UniFi Network Application using Layer-2 discovery (or manual SSH `set-inform` fallback).
2. Update switch firmware safely and label every physical RJ45 & SFP+ port inside UniFi **Port Manager** for crystal-clear topology visibility.
3. Configure **PoE Safety Hygiene**—explicitly setting **PoE = Off** on Port 4 (the `60W PoE++ Injector` data-only line) and non-PoE server ports while keeping active PoE+ ready on future expansion ports.
4. Verify live **2.5 GbE** link speed negotiation to the `UNAS 2`, **1 GbE** links to `homelab-pi`/`homelab-gpu`, switch chassis temperature, and STP/RSTP loop protection.

## 2.2.1 Conceptual Background: Switch Adoption, `set-inform`, and PoE Port Hygiene

### 1. How UniFi Switch Adoption Works Under the Hood
When a factory-default `USW-Enterprise-8-PoE` powers on, it broadcasts Layer-2 discovery packets on UDP port `10001` and listens for a UniFi Network controller on the same subnet (`10.0.0.0/24`).
- **Auto-Discovery:** Because we exposed `10001:10001/udp` and `8080:8080/tcp` on `homelab-gpu` (`10.0.0.201`), your switch should appear automatically under **UniFi Devices** as **`Ready to Adopt`**.
- **Explicit SSH `set-inform` (When Docker Bridge Masks Broadcasts):** Sometimes Docker's default bridge NAT swallows UDP broadcasts on `10001`. If the switch does not appear automatically after 60 seconds, we simply SSH directly into the factory-default switch (`ssh ubnt@10.0.0.203`, default password `ubnt`) and issue `set-inform http://10.0.0.201:8080/inform`. The switch immediately registers with your controller and swaps its SSH credentials to your controller's secure Device SSH Authentication key!

### 2. Why Configure Explicit PoE Hygiene on Port 4 (`UNAS 2` Injector Line)?
Even though `USW-Enterprise-8-PoE` uses **active** 802.3af/at negotiation (so it will not blindly inject 48V into your router, Pi, laptop, or the PoE++ injector's `LAN In` port), disabling PoE in software on non-PoE ports is an enterprise networking best practice:
- **Zero Accidental PoE Interaction with Inline Injectors:** Explicitly setting **PoE = Off** on **Port 4** guarantees that the switch port operates strictly as a pure **2.5 GbE data interface** feeding your external 60W PoE++ injector.
- **Instant Visual Clarity in Port Manager:** At a glance, you immediately see which ports are pure data trunks (Ports 1–4) vs. which ports are actively delivering PoE+ power (e.g., if you add a UniFi Wi-Fi 7 AP or a PoE-powered Raspberry Pi HAT on Port 5 later!).
- **Live Thermal & Link Telemetry:** Once adopted, the UniFi dashboard tracks the fanless switch's internal CPU/ASIC temperature (`°C`), PoE wattage budget (`0W / 120W`), real-time Rx/Tx throughput per port, and color-coded link speeds (**Blue/Cyan = 2.5 GbE**, **Green = 1 GbE**, **White/Purple = 10G SFP+**).

## 2.2.2 Step-by-Step Laboratory Guide

### Task 2.2.1: Adopt `USW-Enterprise-8-PoE` (`10.0.0.203`) into UniFi Network Application
1. Open your UniFi Network Web UI at `https://10.0.0.201:8443` and click the **UniFi Devices** icon in the left sidebar.
2. Check if **`USW-Enterprise-8-PoE`** (`10.0.0.203`) is already listed as **Click to Adopt**:
   - **If visible:** Click **Click to Adopt** and proceed to Step 4!
   - **If NOT visible (Docker bridge broadcast isolation):** SSH directly into the switch from `homelab-gpu` using Ubiquiti's factory-default credentials (`ubnt` / `ubnt`):
     ```bash
     # From homelab-gpu (or your workstation):
     ssh ubnt@10.0.0.203
     ```
     At the `USW-Enterprise-8-PoE-US.#` prompt, point the switch directly to your controller's Inform URL:
     ```text
     set-inform http://10.0.0.201:8080/inform
     ```
     Now refresh **UniFi Devices** in your browser—`USW-Enterprise-8-PoE` will appear immediately! Click **Click to Adopt**, and if the status stays in *Adopting*, run `set-inform http://10.0.0.201:8080/inform` one more time to confirm the handshake.
3. Wait 1–2 minutes for the switch status to turn solid green (**Online**). Notice that the front **1.3" color LCM screen** on the switch automatically updates to display your live throughput and port state!
4. **Update Firmware (Recommended):** If an **Update** badge appears next to `USW-Enterprise-8-PoE`, click **Update** now (the switch will reboot for ~2 minutes; all LAN connections will restore automatically).

### Task 2.2.2: Label All Switch Ports, Apply PoE Hygiene & Verify Telemetry
1. In **UniFi Devices**, click **`USW-Enterprise-8-PoE`** $\rightarrow$ click **Port Manager** (in the right slide-out drawer).
2. Configure clear names and PoE hygiene for Ports 1–8:

| Port | Port Name | Link Speed (Expected) | PoE Setting | Rationale |
| :---: | :--- | :---: | :---: | :--- |
| **Port 1** | `Uplink - Home Router (10.0.0.1)` | 1 GbE / 2.5 GbE | **Off** | Non-PoE router LAN port |
| **Port 2** | `Node 1 - homelab-pi (10.0.0.200)` | **1 GbE** | **Off** *(unless using Pi PoE+ HAT)* | Pure data to Raspberry Pi 4 |
| **Port 3** | `Node 2 - homelab-gpu (10.0.0.201)` | **1 GbE** *(or 2.5 GbE)* | **Off** | Pure data to 2019 ROG Zephyrus |
| **Port 4** | `Node 3 - UNAS 2 via 60W PoE++ Injector (10.0.0.202)` | **2.5 GbE** | **Off** | Pure 2.5GbE data into injector `LAN In` |
| **Ports 5–8** | `Spare 2.5GbE PoE+` | Unplugged | **PoE/PoE+** *(Default)* | Ready for future PoE+ APs, cameras, or nodes |
| **Ports 9–10** | `Spare 10G SFP+` | Unplugged | N/A | Ready for future 10G DAC / fiber uplinks |

3. Click **Apply Changes** after editing each port (or batch-select Ports 1–4 and set **PoE** $\rightarrow$ **Off** simultaneously).
4. In the switch overview panel, inspect and record:
   - **Port 4 Negotiated Speed:** Confirm **`2.5 GbE`**!
   - **RSTP (Spanning Tree):** Confirm **RSTP** is enabled under **Settings** (prevents accidental broadcast storms if a patch cable is ever looped).
   - **Temperature:** Check the internal fanless chassis temperature readout (typically `45°C–65°C` under light load—completely normal for passively cooled 2.5GbE silicon).
5. **Security Best Practice:** Under **Settings $\rightarrow$ System $\rightarrow$ Advanced $\rightarrow$ Device Authentication**, enable **Device SSH Authentication** with a unique strong password (or paste your `~/.ssh/id_ed25519_homelab.pub` public key!) so the switch never uses factory `ubnt/ubnt` credentials again.

### 🖼️ Task 2.2.2 Verification: Adopted `USW-Enterprise-8-PoE` Port Manager & 2.5GbE Telemetry

**Verification Objective:** Capture a screenshot of the UniFi **Port Manager** view for your adopted `USW-Enterprise-8-PoE` showing labeled Ports 1–4, **2.5 GbE** link negotiation on Port 4 (`UNAS 2`), and **PoE = Off** hygiene applied to Ports 1–4.

---

> ### 📷 User Verification Screenshot Submission Box
> *Attach your milestone verification screenshot below (drag & drop an image or use a Markdown image link):*
> 
> ```markdown
> <!-- Insert your screenshot here: -->
> 
> ```

## 🏁 Module Checkpoint & Reflection

Before proceeding to the next guide, confirm the following checklist:

- [ ] `USW-Enterprise-8-PoE` (`10.0.0.203`) adopted and showing **Online** in UniFi Network Application (`https://10.0.0.201:8443`).
- [ ] Switch firmware up to date and custom Device SSH Authentication key/password configured.
- [ ] Ports 1–4 cleanly labeled in **Port Manager** with **PoE = Off** applied to Ports 1–4 (including Port 4 feeding the 60W PoE++ injector).
- [ ] Live **2.5 GbE** link speed verified on Port 4 (`UNAS 2`) and healthy fanless chassis temperature confirmed.
- [ ] Verification screenshot captured and attached above.

Proceed to [Module 3.1: UniFi Drive Onboarding, RAID 1 Storage Pool, Btrfs Snapshots & NFS/SMB Shares](../03_unas2_storage_pool_and_linux_mounts/03_1_unifi_drive_raid1_and_shares.ipynb).